In [ ]:
import os
from pathlib import Path

import torch
import wandb

from datasets.mnist import MNISTSampler
from models.config import load_config
from models.flow import FlowModel
from training.path import GaussianConditionalProbabilityPath, LinearAlpha, LinearBeta
from training.trainer import FlowTrainer

persistent_root = Path(os.getenv("DIFFUSION_DATA_ROOT", "/workspace-global/Diffusion-data"))
data_root = persistent_root / "datasets" / "mnist"
checkpoints_dir = persistent_root / "checkpoints"
samples_dir = persistent_root / "samples"
wandb_dir = persistent_root / "outputs" / "wandb"
for path in (data_root, checkpoints_dir, samples_dir, wandb_dir):
    path.mkdir(parents=True, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("device", device)

CONFIG = "configs/mnist.yaml"
cfg = load_config(CONFIG)
flow = FlowModel.from_config(CONFIG).to(device)

xt = torch.randn(
    cfg["test"]["batch_size"],
    cfg["unet"]["in_channels"],
    cfg["test"]["image_size"],
    cfg["test"]["image_size"],
    device=device,
)
t = torch.rand(xt.shape[0], device=device)
print("u(x, t)", tuple(flow(xt, t).shape))

In [ ]:
def make_path(split: str) -> GaussianConditionalProbabilityPath:
    return GaussianConditionalProbabilityPath(
        p_data=MNISTSampler(root=str(data_root), split=split),
        p_simple_shape=[1, 32, 32],
        alpha=LinearAlpha(),
        beta=LinearBeta(),
    ).to(device)

train_path = make_path("train")
val_path = make_path("val")

trainer = FlowTrainer(path=train_path, model=flow, val_path=val_path)
run = wandb.init(
    project="mnist-flow-matching",
    dir=str(wandb_dir),
    config={
        "num_steps": 5000,
        "batch_size": 64,
        "learning_rate": 1e-3,
        "model": cfg,
    },
)
try:
    history = trainer.train(
        num_steps=5000,
        device=device,
        lr=1e-3,
        batch_size=64,
        ckpt_path=checkpoints_dir / "mnist_flow.pt",
        checkpoint_every=50,
        val_every=50,
        val_batches=8,
        plot_every=50,
        n_plot_images=10,
        n_plot_steps=10,
        samples_dir=samples_dir,
        show_plots=False,
        wandb_run=run,
    )
finally:
    run.finish()